# 07 - Hooks, permissions, and guardrails

A prompt rule is a strong suggestion. A hook is a wall.

Hooks are callbacks the SDK runs at fixed points in the agent loop. They run in
**your** process, not in Claude's context window — so they cost no tokens, they
see everything, and they cannot be argued with by a cleverly worded ticket.

Hooks run **first** in the permission evaluation, before deny rules, before
allow rules, before `can_use_tool`. A hook deny applies even in
`bypassPermissions`.

**Docs:** [Hooks](https://code.claude.com/docs/en/agent-sdk/hooks) ·
[Permissions](https://code.claude.com/docs/en/agent-sdk/permissions)

In [3]:
# --- boot cell: run this first in every notebook -------------------------
import os, sys, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

# The SDK reads ANTHROPIC_API_KEY from the process environment.
# It does NOT parse .env for you, so we do it here.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and add your key."
)

MODEL = os.environ.get("COURSE_MODEL") or None   # None = SDK default
print("key loaded, model =", MODEL or "(SDK default)")

key loaded, model = claude-haiku-5-5


In [4]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, HookMatcher,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)

SCRATCH = ROOT / "workspace" / "_scratch"
(SCRATCH / "sandbox").mkdir(parents=True, exist_ok=True)
(SCRATCH / "notes.txt").write_text("alpha\nbeta\n", encoding="utf-8")

async def run(prompt, **opts):
    opts.setdefault("cwd", str(SCRATCH))
    opts.setdefault("setting_sources", [])
    opts.setdefault("max_turns", 8)
    opts.setdefault("model", MODEL)
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock) and b.text.strip():
                        print(f"   ... {b.text.strip()[:150]}")
                    elif isinstance(b, ToolUseBlock):
                        print(f"   TOOL {b.name} {str(b.input)[:70]}")
            elif isinstance(m, ResultMessage):
                print(f"   ==> {m.subtype} ${m.total_cost_usd or 0:.4f}")
    except Exception as exc:
        print(f"   raised: {str(exc)[:90]}")

print("ready")

ready


## 7.1 The hook events (Python SDK)

| Event | Fires | Typical use |
|---|---|---|
| `PreToolUse` | before a tool executes | **block**, **modify input**, audit |
| `PostToolUse` | after a tool returns | audit output, append context, replace output |
| `PostToolUseFailure` | a tool call failed | log or handle errors |
| `UserPromptSubmit` | a prompt is submitted | **inject context** into the prompt |
| `Stop` | the agent finishes | validate the result, persist state |
| `SubagentStart` / `SubagentStop` | a subagent spawns / completes | track parallel work |
| `PreCompact` | before compaction | archive the full transcript |
| `PermissionRequest` | a call needs a decision | custom permission handling |
| `Notification` | agent status message | pipe to Slack, PagerDuty |

The TypeScript SDK has more events (`SessionStart`, `SessionEnd`,
`MessageDisplay`, `TaskCompleted`, and others). The table above is what Python
supports today.

**Docs:** [Available hooks](https://code.claude.com/docs/en/agent-sdk/hooks#available-hooks)

## 7.2 The shape of a hook

**Registration** — event name → list of matchers → list of callbacks:

```python
hooks={
    "PreToolUse": [
        HookMatcher(hooks=[log_everything]),                    # no matcher: every tool
        HookMatcher(matcher="Write|Edit", hooks=[guard_writes]), # regex on tool name
        HookMatcher(matcher="^mcp__", hooks=[audit_mcp]),        # every MCP tool
    ],
    "SubagentStop": [HookMatcher(hooks=[track])],
}
```

**Callback** — three arguments, always:

```python
async def my_hook(input_data, tool_use_id, context):
    ...
    return {}        # {} means "allow, unchanged"
```

- `input_data` — a dict. Always has `session_id`, `cwd`, `hook_event_name`.
  `PreToolUse` adds `tool_name` and `tool_input`. Inside a subagent it also
  carries `agent_id` and `agent_type`.
- `tool_use_id` — correlates `PreToolUse` with the matching `PostToolUse`.
- `context` — reserved in Python.

**Return value** — `{}` to allow, or `hookSpecificOutput` to change something:

```python
return {"hookSpecificOutput": {
    "hookEventName": "PreToolUse",
    "permissionDecision": "deny",         # allow | deny | ask | defer
    "permissionDecisionReason": "why",    # Claude reads this and adapts
    # "updatedInput": {...},              # rewrite the arguments
}}
```

When several hooks apply: **deny > defer > ask > allow**. Any deny blocks.

## 7.3 Hook 1 — audit every tool call

No matcher, so it fires for every tool from every agent. This is the cheapest
observability you will ever add: no tokens, complete coverage.

In [5]:
AUDIT = []

async def audit_all(input_data, tool_use_id, context):
    AUDIT.append({
        "tool": input_data.get("tool_name"),
        "agent": input_data.get("agent_type") or "main",
        "args": str(input_data.get("tool_input", {}))[:60],
    })
    return {}     # allow, unchanged


AUDIT.clear()
await run(
    "Read notes.txt, then count how many lines it has using bash.",
    tools=["Read", "Bash", "Glob"],
    allowed_tools=["Read", "Bash", "Glob"],
    permission_mode="dontAsk",
    hooks={"PreToolUse": [HookMatcher(hooks=[audit_all])]},
)

print("\naudit trail:")
for entry in AUDIT:
    print(f"   {entry['agent']:<6} {entry['tool']:<8} {entry['args']}")

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


   TOOL Bash {'command': 'pwd && ls -la', 'description': 'Show working directory an
   TOOL Read {'file_path': '/Users/vijay/Desktop/TCS_Mexico/workspace/TCS_FDE_Train
   TOOL Bash {'command': 'wc -l /Users/vijay/Desktop/TCS_Mexico/workspace/TCS_FDE_T
   ... `notes.txt` has **2 lines** by `wc -l`. The file contains `alpha` and `beta`, each ending in a newline (11 bytes total).

The Read tool showed a third
   ==> success $0.0023

audit trail:
   main   Bash     {'command': 'pwd && ls -la', 'description': 'Show working di
   main   Read     {'file_path': '/Users/vijay/Desktop/TCS_Mexico/workspace/TCS
   main   Bash     {'command': 'wc -l /Users/vijay/Desktop/TCS_Mexico/workspace


## 7.4 Hook 2 — block an operation

The canonical example: never let the agent touch `.env`.

Note the `permissionDecisionReason`. Claude reads it and adapts. A good reason
turns a block into a correction; a bad one turns it into a retry loop.

In [6]:
async def protect_secrets(input_data, tool_use_id, context):
    path = str(input_data.get("tool_input", {}).get("file_path", ""))
    name = path.rsplit("/", 1)[-1].rsplit("\\", 1)[-1]

    if name in (".env", ".env.local") or name.endswith(".pem"):
        print(f"   [HOOK] blocked write to {name}")
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": (
                "Secret files are off limits. Write the example values to "
                "config.example.txt instead and tell the user what you did."
            ),
        }}
    return {}


await run(
    "Create a .env file with placeholder database settings.",
    tools=["Write", "Read", "Glob"],
    allowed_tools=["Write", "Read", "Glob"],
    permission_mode="acceptEdits",     # note: the hook still wins
    hooks={"PreToolUse": [HookMatcher(matcher="Write|Edit", hooks=[protect_secrets])]},
)
print("\n.env created?", (SCRATCH / ".env").exists())

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


   TOOL Glob {'pattern': '.env*'}
   TOOL Glob {'pattern': '**/*{database,db,config}*.{js,ts,py,rb,go,yml,yaml,json,t
   TOOL Glob {'pattern': '*'}
   TOOL Write {'file_path': '.env', 'content': '# Database configuration (placeholde
   [HOOK] blocked write to .env
   ... I couldn't create the `.env` file. The write was blocked because the tool treats `.env` files as secret files. I didn't try another filename, since th
   ... My last message had stray closing tags in the text, which is why it looked malformed. It wasn't a real tool call.

I'm not going to retry the `.env` w
   ==> success $0.0027

.env created? False


`permission_mode="acceptEdits"` would have auto-approved that write.
The hook denied it anyway, because hooks are evaluated *before* the permission
mode. This is the property that makes hooks the right place for rules that must
always hold.

## 7.5 Hook 3 — rewrite the arguments

`updatedInput` replaces the tool's arguments before it runs. Here every write is
redirected into a `sandbox/` subdirectory.

Pair `updatedInput` with `permissionDecision: "allow"` to auto-approve the
rewritten call. Always return a **new** dict rather than mutating `tool_input`.

In [7]:
async def redirect_to_sandbox(input_data, tool_use_id, context):
    if input_data.get("hook_event_name") != "PreToolUse":
        return {}

    tool_input = input_data.get("tool_input", {})
    path = str(tool_input.get("file_path", ""))
    if not path or "/sandbox/" in path.replace("\\", "/"):
        return {}

    name = path.rsplit("/", 1)[-1].rsplit("\\", 1)[-1]
    new_path = str(SCRATCH / "sandbox" / name)
    print(f"   [HOOK] redirect {name} -> sandbox/{name}")

    return {"hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "allow",
        "updatedInput": {**tool_input, "file_path": new_path},
    }}


await run(
    "Write a file called report.md containing a one-line summary of notes.txt.",
    tools=["Write", "Read", "Glob"],
    allowed_tools=["Write", "Read", "Glob"],
    permission_mode="acceptEdits",
    hooks={"PreToolUse": [HookMatcher(matcher="Write", hooks=[redirect_to_sandbox])]},
)

print("\nin sandbox/:", [p.name for p in (SCRATCH / "sandbox").glob("*")])
print("at top level:", (SCRATCH / "report.md").exists())

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


   TOOL Glob {'pattern': '**/notes.txt'}
   TOOL Read {'file_path': 'notes.txt'}
   TOOL Write {'file_path': 'report.md', 'content': 'notes.txt contains two lines: "
   [HOOK] redirect report.md -> sandbox/report.md
   ... I created `report.md` in the same directory as `notes.txt`. It contains this one-line summary:

> notes.txt contains two lines: "alpha" and "beta".
   ==> success $0.0014

in sandbox/: ['report.md']
at top level: False


## 7.6 Hook 4 — inject context into every prompt

`UserPromptSubmit` fires when a prompt is submitted, and `additionalContext` is
appended to it. Use it for facts that change between runs and must not be
compacted away: the current date, the on-call engineer, a feature flag, the
caller's tenant.

In [8]:
from datetime import datetime, timezone

async def inject_runtime_context(input_data, tool_use_id, context):
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    return {"hookSpecificOutput": {
        "hookEventName": "UserPromptSubmit",
        "additionalContext": (
            f"[runtime context] Current time: {now}. "
            f"On-call engineer: J. Okoye. Deploy freeze: active until Monday."
        ),
    }}


await run(
    "Should I ship a hotfix right now? One sentence, and say who to contact.",
    tools=[],
    hooks={"UserPromptSubmit": [HookMatcher(hooks=[inject_runtime_context])]},
    max_turns=1,
)

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


   ... No, not while the deploy freeze is active until Monday. Contact on-call engineer J. Okoye if the issue is severe enough to need an emergency exception
   ==> success $0.0012


## 7.7 `can_use_tool`: the interactive approval callback

Hooks are for rules you can express in code. `can_use_tool` is for decisions a
**human** makes at run time — the last step of the permission flow.

```python
from claude_agent_sdk import PermissionResultAllow, PermissionResultDeny

async def approve(tool_name, input_data, context):
    if tool_name == "Bash" and "rm " in input_data.get("command", ""):
        return PermissionResultDeny(message="No deletions.", interrupt=True)
    # In a real app: render a prompt in your UI and await the answer.
    return PermissionResultAllow(updated_input=input_data)

options = ClaudeAgentOptions(can_use_tool=approve, permission_mode="default")
```

**The trap:** auto-approved tools never reach the callback. If a tool is
approved by an allow rule or by `acceptEdits`/`bypassPermissions`, your callback
is skipped for it. A check that must run on *every* call belongs in a
`PreToolUse` hook, not here.

## 7.8 Which control for which job

| Need | Use | Why |
|---|---|---|
| The tool must not exist | `tools=[...]` / bare `disallowed_tools` | cheapest; Claude never tries |
| Block a command pattern | `disallowed_tools=["Bash(rm *)"]` | declarative, no code |
| A rule that must **always** hold | **`PreToolUse` hook** | runs first, beats every mode |
| Rewrite arguments | `PreToolUse` hook + `updatedInput` | the only place to do it |
| A human decides | `can_use_tool` | that is what it is for |
| Audit everything | `PreToolUse` hook, no matcher | complete, zero tokens |
| Inject changing facts | `UserPromptSubmit` hook | survives compaction |

**A design rule worth keeping:** anything you can decide deterministically,
decide in code. Reserve the model for judgement. "Never confirm a deletion" is
not judgement — it is a regex.

## 7.9 Async hooks

If a hook only has a side effect — logging, metrics, a webhook — do not make the
agent wait for it:

```python
async def notify(input_data, tool_use_id, context):
    asyncio.create_task(send_to_datadog(input_data))
    return {"async_": True, "asyncTimeout": 30000}   # note the trailing underscore
```

`async_` in Python (`async` is a keyword), `async` in TypeScript. An async hook
**cannot** block, modify, or inject — the agent has already moved on.

## 7.10 Exercises

1. Write a `PostToolUse` hook that logs every `Read` and the size of what came
   back. Which file dominated the context?
2. Write a `PreToolUse` hook on `Bash` that denies any command containing a pipe
   to `sh` or `bash`. Test it with a plausible-looking install command.
3. Write a `Stop` hook that saves `session_id` and a timestamp to a JSON file.
4. Combine two hooks on `PreToolUse` — one that redirects and one that denies —
   and confirm the deny wins regardless of registration order.
5. **The real one:** open
   [notebook 11](11_build_multi_agent_triage.ipynb) and read section 11.4,
   where `enforce_hard_stops` is built and fired offline. Four separate prompts
   there already say "never confirm a deletion". Explain in one sentence why the
   hook still earns its place.

In [9]:
# Scratch space for the exercises.

## Next

[**08 - Subagents and multi-agent systems**](08_subagents_and_multi_agent.ipynb)
— splitting one agent into a team, and when that is worth the cost.

---

### References

- [Control execution with hooks](https://code.claude.com/docs/en/agent-sdk/hooks)
- [Configure permissions](https://code.claude.com/docs/en/agent-sdk/permissions)
- [Handle approvals and user input](https://code.claude.com/docs/en/agent-sdk/user-input)
- [Claude Code hooks reference](https://code.claude.com/docs/en/hooks)